# AIDA Nanopublication Generator

Creates AIDA (Atomic, Independent, Declarative, Absolute) sentence nanopublications from a JSON configuration file.

**Template:** [AIDA Sentence Template](https://w3id.org/np/RALmXhDw3rHcMveTgbv8VtWxijUHwnSqhCmtJFIPKWVaA)

## AIDA Sentences
- **A**tomic: A sentence describing one thought that cannot be further broken down
- **I**ndependent: A sentence that can stand on its own, without external references
- **D**eclarative: A complete sentence ending with a full stop that could be true or false
- **A**bsolute: A sentence describing the core of a claim ignoring uncertainty

## Install dependencies if needed

In [1]:
# Install dependencies (uncomment if needed)
# !pip install nanopub rdflib

## Setup
- profile.yml contains information for authoring and signing nanopublications (orcid/Software-Agent and public/private keys)
- Prior to the execution of this notebook, you need either to create and user personal orcid names & keys or a softare-agent:
  
  * To create nanpublications signed with your personal ORCID, signed up at https://nanodash.knowledgepixels.com and create a profile (update it accordingly!):
```  
orcid_id: https://orcid.org/0000-XXXX-XXXX-XXXX
name: FirstName LastName
public_key: /path/to/id_rsa_pub.pem
private_key: /path/to/id_rsa
introduction_nanopub_uri:
``` 

  * To create a software agent, use the following nanopublication template: https://w3id.org/np/RAbn04KkfbV5PK2UDGkp-j7RUghs_y75DL4qWl_8zQQ3w
``` 
name: claude-ai-agent
orcid_id: https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent
public_key: /Users/annef/Documents/ScienceLive/ai-profile/aiagent_rsa.pub
private_key: /Users/annef/Documents/ScienceLive/ai-profile/aiagent_rsa
``` 
- inputs/scientific_claims.json contains one or more claims

In [2]:
# Path to your AIDA JSON config file
PROFILE_PATH = "/home/afouilloux/ScienceLive/ai-profile/profile.yml"  # Set to your profile.yml path
#PROFILE_PATH = "/home/afouilloux/ScienceLive/annefou-profile/profile.yml"
CONFIG_FILE = "../inputs/scientific_claims.json"
OUTPUT_DIR = "../outputs/"

## Import Python Packages

In [3]:
import json
import urllib.parse
from pathlib import Path
from datetime import datetime, timezone
from rdflib import Dataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from nanopub import Nanopub, NanopubConf, load_profile

In [4]:
profile = load_profile(PROFILE_PATH)
print(f"Loaded profile: \n{profile.name}\n{profile.orcid_id}")

Loaded profile: 
claude-ai-agent
https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent


## AIDA nanopublication Setup

In [5]:
PUBLISHER = "https://sciencelive4all.org/"

# Namespaces (matching Nanodash)
TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")
HYCL = Namespace("http://purl.org/petapico/o/hycl#")
CITO = Namespace("http://purl.org/spar/cito/")
SCHEMA = Namespace("http://schema.org/")
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")
AIDA = Namespace("http://purl.org/aida/")

# Template URIs
AIDA_TEMPLATE = URIRef("https://w3id.org/np/RALmXhDw3rHcMveTgbv8VtWxijUHwnSqhCmtJFIPKWVaA")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RA0J4vUn_dekg-U1kK3AOEt02p9mT2WO03uGxLDec1jLw")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

# AIDA-specific constants
HYCL_AIDA_SENTENCE = URIRef("http://purl.org/petapico/o/hycl#AIDA-Sentence")
HYCL_NS = URIRef("http://purl.org/petapico/o/hycl")

print("Setup complete")

Setup complete


## Load config file to create AIDA sentences

In [6]:
# Load configuration
print(f"Loading: {CONFIG_FILE}")

with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    config = json.load(f)

Loading: ../inputs/scientific_claims_NOTALL.json


## Extract metadata from profile & config

In [7]:
# Extract metadata
metadata = config.get('metadata', {})


AUTHOR_ORCID = metadata.get('creator_orcid')
AUTHOR_NAME = metadata.get('creator_name')
IS_PART_OF = metadata.get('is_part_of', {})

# Validate
errors = []
print(AUTHOR_ORCID, AUTHOR_NAME)
if not AUTHOR_ORCID or not AUTHOR_NAME:
    AUTHOR_ORCID = profile.orcid_id
    AUTHOR_NAME = profile.name
    metadata["creator_orcid"] = AUTHOR_ORCID
    metadata["creator_name"] = AUTHOR_NAME

if not AUTHOR_ORCID:
    errors.append("URI of the author is missing")
if not AUTHOR_NAME:
    errors.append("Name of the author is missing")
if not config.get('nanopublications'):
    errors.append("nanopublications list is required")

if errors:
    print("❌ Validation errors:")
    for e in errors:
        print(f"   - {e}")
    raise ValueError("Please fix the errors in your JSON file")

print(f"✓ Loaded {len(config['nanopublications'])} AIDA nanopubs to generate")
print(f"✓ Author: {AUTHOR_NAME} ({AUTHOR_ORCID})")
if IS_PART_OF:
    print(f"✓ Part of: {IS_PART_OF.get('label', IS_PART_OF.get('uri'))}")

None None
✓ Loaded 8 AIDA nanopubs to generate
✓ Author: claude-ai-agent (https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent)
✓ Part of: None


## Function to build nanopublications

In [8]:
def create_aida_nanopub(np_config, metadata):
    """
    Create an AIDA sentence nanopublication using rdflib Dataset.
    
    Args:
        np_config: dict with AIDA sentence configuration
        metadata: dict with author and part_of info
    
    Returns:
        tuple: (Dataset, label)
    """
    # CRITICAL: Use temporary namespace that gets replaced when signing
    
    this_np = URIRef("http://purl.org/nanopub/temp/np/")
    head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
    assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
    provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
    pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")

    if metadata['creator_orcid'][0:4] != "http":
        author_uri = ORCID[metadata['creator_orcid']]
    else:
        # When creating the author_uri, ensure it's a URIRef:
        author_uri = URIRef(metadata['creator_orcid'])

    print(author_uri)
    
    # Create AIDA sentence URI (URL-encoded)
    aida_sentence = np_config['aida_sentence']
    aida_uri = URIRef(f"http://purl.org/aida/{urllib.parse.quote(aida_sentence, safe='')}")
    
    # Create Dataset
    ds = Dataset()
    
    # Bind prefixes
    ds.bind("this", "http://purl.org/nanopub/temp/np/")
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("xsd", XSD)
    ds.bind("rdfs", RDFS)
    ds.bind("orcid", ORCID)
    ds.bind("prov", PROV)
    ds.bind("foaf", FOAF)
    ds.bind("hycl", HYCL)
    ds.bind("cito", CITO)
    ds.bind("schema", SCHEMA)
    ds.bind("skos", SKOS)
    
    # HEAD graph
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))
    
    # ASSERTION graph
    assertion = ds.graph(assertion_graph)
    assertion.add((aida_uri, RDF.type, HYCL_AIDA_SENTENCE))
    
    # Related publication
    if np_config.get('related_publication'):
        pub = np_config['related_publication']
        pub_uri = URIRef(pub if pub.startswith('http') else f"https://doi.org/{pub}")
        assertion.add((aida_uri, CITO.obtainsSupportFrom, pub_uri))
    
    # Related dataset
    if np_config.get('related_dataset'):
        assertion.add((aida_uri, CITO.obtainsSupportFrom, URIRef(np_config['related_dataset'])))
    
    # Topics
    topics = np_config.get('topics', [])
    for topic in topics:
        if topic.get('uri'):
            assertion.add((aida_uri, SCHEMA.about, URIRef(topic['uri'])))
    
    # Related project
    if np_config.get('related_project'):
        assertion.add((aida_uri, SKOS.related, URIRef(np_config['related_project'])))
    
    # isPartOf (in assertion for AIDA)
    is_part_of = metadata.get('is_part_of', {})
    if is_part_of.get('uri'):
        assertion.add((aida_uri, DCT.isPartOf, URIRef(is_part_of['uri'])))
    
    # PROVENANCE graph
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))
    
    # PUBINFO graph
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author_uri, FOAF.name, Literal(metadata['creator_name'])))
    
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    
    # Nanopub types
    pubinfo.add((this_np, NPX.hasNanopubType, HYCL_AIDA_SENTENCE))
    pubinfo.add((this_np, NPX.hasNanopubType, HYCL_NS))
    
    # Introduces
    pubinfo.add((this_np, NPX.introduces, aida_uri))
    
    # Label
    label_text = aida_sentence[:100] + "..." if len(aida_sentence) > 100 else aida_sentence
    label = f"AIDA sentence: {label_text}"
    pubinfo.add((this_np, RDFS.label, Literal(label)))
    
    # Template references
    pubinfo.add((this_np, NT.wasCreatedFromTemplate, AIDA_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    
    # Add topic labels in pubinfo (for Nanodash display)
    for topic in topics:
        if topic.get('uri') and topic.get('label'):
            pubinfo.add((URIRef(topic['uri']), NT.hasLabelFromApi, Literal(topic['label'])))
    
    return ds, label

print("✓ Function defined")

✓ Function defined


## Create output directory and Generate Nanopublication files

In [9]:
# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Generate all nanopublications
generated_files = []

n = 1
for np_config in config['nanopublications']:
    # Create nanopub
    if not "related_publication" in np_config:
        np_config["related_publication"] = metadata.get("source_paper").get("doi")
        
    ds, label = create_aida_nanopub(np_config, metadata)
    
    # Save to file
    filename = np_config['related_publication'].replace(".","_").replace("/","_") + "_" + str(n)
    print(filename)
    output_file = Path(OUTPUT_DIR) / f"{filename}.trig"
    ds.serialize(destination=str(output_file), format='trig')
    generated_files.append(output_file)
    
    print(f"✓ Generated: {output_file}")
    n = n + 1

print(f"\nTotal generated: {len(generated_files)} nanopublications")

https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent
10_3390_urbansci9020038_1
✓ Generated: ../outputs/10_3390_urbansci9020038_1.trig
https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent
10_3390_urbansci9020038_2
✓ Generated: ../outputs/10_3390_urbansci9020038_2.trig
https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent
10_3390_urbansci9020038_3
✓ Generated: ../outputs/10_3390_urbansci9020038_3.trig
https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent
10_3390_urbansci9020038_4
✓ Generated: ../outputs/10_3390_urbansci9020038_4.trig
https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent
10_3390_urbansci9020038_5
✓ Generated: ../outputs/10_3390_urbansci9020038_5.trig
https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent
10_3390_urbansci9020038_6
✓ Generated: ../outputs/10_3390_urbansci9020038_6.trig
https://w3id.org/np/RA

## Preview and output

In [10]:
# Preview first generated nanopublication
if generated_files:
    print(f"Preview of {generated_files[0]}:\n")
    print("=" * 80)
    with open(generated_files[0], 'r') as f:
        print(f.read())

Preview of ../outputs/10_3390_urbansci9020038_1.trig:

@prefix cito: <http://purl.org/spar/cito/> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix hycl: <http://purl.org/petapico/o/hycl#> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema1: <http://schema.org/> .
@prefix sub: <http://purl.org/nanopub/temp/np/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

sub:provenance {
    sub:assertion prov:wasAttributedTo <https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent> .
}

sub:pubinfo {
    sub: rdfs:label "AIDA sentence: The urban microclimate of Avenida dos Aliados and Praça da Liberdade exhibits a temperature range of..." ;
        dct:created "2026-03-01T11:14:28.000Z"^^xsd:dateTime ;
        dct:

## Sign and publish (optional)

In [15]:
PUBLISH = False  # Set to True when ready to publish

In [14]:
if PUBLISH:
    conf = NanopubConf(profile=profile)
    
    for trig_file in generated_files:
        np_obj = Nanopub(rdf=trig_file, conf=conf)
        np_obj.sign()
        
        signed_path = trig_file.with_suffix('.signed.trig')
        np_obj.store(signed_path)
        print(f"✓ Signed: {signed_path}")
        
        np_obj.publish()
        print(f"✓ Published: {np_obj.source_uri}")
else:
    print("Publishing disabled. Set PUBLISH = True to enable.")
    print("\nManual signing:")
    for f in generated_files:
        print(f"  nanopub sign {f}")
        print(f"  nanopub publish {f.stem}.signed.trig")

✓ Signed: ../outputs/10_3390_urbansci9020038_1.signed.trig
✓ Published: http://purl.org/np/RA54Dg18LsfaU9Vu9tcepTzjweyG1zG_bK41m3YOqk1Xk
✓ Signed: ../outputs/10_3390_urbansci9020038_2.signed.trig
✓ Published: http://purl.org/np/RAnPB76ppabKhm40ch7IsFTrYRTbwd4_DXt_HELi2QujM
✓ Signed: ../outputs/10_3390_urbansci9020038_3.signed.trig
✓ Published: http://purl.org/np/RAj79pJB5dugac72iVkUwUdMrtmOn4ojx-NfwqXz7Ms4Q
✓ Signed: ../outputs/10_3390_urbansci9020038_4.signed.trig
✓ Published: http://purl.org/np/RADTumRL5X6f_UUxRykf1W-C9V79NHkpGAbcpn_7BICMg
✓ Signed: ../outputs/10_3390_urbansci9020038_5.signed.trig
✓ Published: http://purl.org/np/RAfAwk0F4VqdU3GRtq6l0fyd5MXjoT-kbDIA5rU-dL6Fk
✓ Signed: ../outputs/10_3390_urbansci9020038_6.signed.trig
✓ Published: http://purl.org/np/RAYEJaEBxQErjVe3lDT2eMq7Xe9duUPnsjTAcfw_luk3g
✓ Signed: ../outputs/10_3390_urbansci9020038_7.signed.trig
✓ Published: http://purl.org/np/RA1ABUmzouT1OjxM1eq8YLMbJAKZUR_4uOHi6Kkc8qgWU
✓ Signed: ../outputs/10_3390_urbansci9020